# Module 4 - One AI gateway, free models, and tracing

**Anti-pattern:** every service calls a provider SDK with its own key and model name. When a free model disappears (it will) you edit five codebases.
**Gateway pattern:** every service calls `http://litellm:4000/v1` with an alias (`formation-fast`); the gateway holds keys, routes, retries, falls back and reports.

Open the gateway config next to this notebook: `config/litellm_config.yaml`.

In [ ]:
import os, sys, json, asyncio, time, logging
from pathlib import Path
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)                                   # every path below is relative to the project root
os.environ.setdefault("HF_HOME", str(ROOT / "models" / "huggingface"))
logging.getLogger("neo4j").setLevel(logging.ERROR)
MODEL = os.getenv("FORMATION_MODEL", "formation-fast")   # use "formation-local" if you run LM Studio / Ollama

import httpx, base64
from backend.llm import chat, GatewayError
GATEWAY = os.getenv("LITELLM_URL", "http://localhost:14000/v1")
KEY = os.getenv("LITELLM_MASTER_KEY", "sk-syone-formation-master-key")
print("gateway:", GATEWAY)

## 1. Which free models exist *right now*?
The list changes every few weeks, which is exactly why services use aliases.

In [ ]:
import pandas as pd
r = httpx.get("http://localhost:18000/api/models/free", timeout=20).json()
print("source:", r["source"], "|", r["count"], "models")
df = pd.DataFrame(r["models"])[["id", "context_length", "supports_vision", "supports_tools"]]
display(df.head(12))

## 2. Call every alias through the gateway

In [ ]:
async def probe(alias):
    t0 = time.time()
    try:
        m = await chat([{"role": "user", "content": "Reply with exactly: ready"}], alias, max_tokens=200, timeout=120)
        return alias, "OK", f"{time.time()-t0:.1f}s", (m.get("content") or "").strip()[:30]
    except GatewayError as e:
        return alias, "FAIL", f"{time.time()-t0:.1f}s", str(e)[:90]

rows = [await probe(a) for a in ("formation-fast", "formation-reasoning", "formation-vision", "formation-local")]
display(pd.DataFrame(rows, columns=["alias", "status", "time", "reply / error"]))

### Reading the table
* `OK`: the alias works. `FAIL` with *User not found* = the OpenRouter key in `.env` is still the placeholder; `free-models-per-day` = the daily cap (see section 3).
* `formation-local` works only if LM Studio / Ollama is running on your machine.
* An alias can `FAIL` here only after **all** its failover layers (section 3) have failed.

## 3. Failover: what a gateway is *for*
Free tiers hit limits (20 requests/minute and 50/day on OpenRouter without credit), providers have outages, models disappear. The gateway absorbs that so your application never notices. Three layers, all in `config/litellm_config.yaml`:

| Layer | Setting | What happens |
|---|---|---|
| 1. Deployments in order | several `model_name: formation-fast` entries with `order: 1, 2, 3...` | a failed call is retried on the next provider |
| 2. Cooldown | `allowed_fails: 1`, `cooldown_time: 30` | a deployment that returned 429/5xx is skipped for 30 s: later calls do not even try it |
| 3. Alias fallbacks | `fallbacks: formation-fast -> formation-reasoning -> formation-local` | the whole alias failed: try another model family, then a local model |

Proof without spending quota: the alias `formation-failover-drill` has a primary deployment pointing at a **closed port** (a simulated outage) and a secondary that answers. Call it three times and read the response headers.

In [ ]:
def drill():
    r = httpx.post(f"{GATEWAY}/chat/completions", headers={"Authorization": f"Bearer {KEY}"}, timeout=60,
                   json={"model": "formation-failover-drill", "messages": [{"role": "user", "content": "hi"}]})
    h = r.headers
    return {"status": r.status_code, "answer": r.json()["choices"][0]["message"]["content"][:48],
            "fallbacks": h.get("x-litellm-attempted-fallbacks"), "retries": h.get("x-litellm-attempted-retries"),
            "ms": round(float(h.get("x-litellm-response-duration-ms", 0)), 1)}
display(pd.DataFrame([drill() for _ in range(3)]))

Read it: the **first** call tried the dead primary, failed and was served by the secondary (`fallbacks = 1`, slower). The **next** calls are fast because the dead deployment is in **cooldown** and was not even tried. After 30 seconds the gateway tries it again (self-healing).
Now the real chain: the default alias with every provider failing ends in the application's own safety net.

In [ ]:
try:
    await chat([{"role": "user", "content": "hi"}], "alias-that-does-not-exist", max_tokens=5, timeout=30)
except GatewayError as e:
    print("GatewayError ->", str(e)[:160])
print("\nWhen EVERY layer fails, the pipelines return an EXTRACTIVE answer (retrieved passages, clearly labelled) instead of inventing text.")

## 3b. Virtual keys: hand out access, not provider keys
A gateway issues its **own** keys, limited to chosen model aliases, with a budget, a rate limit and an expiry. The provider key (OpenCode, OpenRouter...) stays inside the gateway. This cell mints a throw-away key (needs the **admin** key in `LITELLM_MASTER_KEY`), uses it the way a trainee would, and deletes it.

In [ ]:
BASE = GATEWAY.rsplit("/v1", 1)[0]
admin = {"Authorization": f"Bearer {KEY}"}
g = httpx.post(f"{BASE}/key/generate", headers=admin, timeout=30,
               json={"key_alias": "nb4-demo", "models": ["formation-fast"], "max_budget": 0.02, "rpm_limit": 3, "duration": "1h"})
if g.status_code != 200:
    print("Needs the gateway ADMIN key in LITELLM_MASTER_KEY (a trainee virtual key cannot mint keys):", g.status_code)
else:
    vkey = g.json()["key"]
    def as_trainee(model):
        r = httpx.post(f"{GATEWAY}/chat/completions", headers={"Authorization": f"Bearer {vkey}"}, timeout=90,
                       json={"model": model, "messages": [{"role": "user", "content": "Reply: ok"}], "max_tokens": 20})
        return {"requested model": model, "HTTP": r.status_code, "meaning": {200: "answered", 403: "not allowed for this key", 429: "rate limit (3/min)", 422: "budget exceeded"}.get(r.status_code, "")}
    rows = [as_trainee(m) for m in ("formation-fast", "formation-reasoning", "openrouter/openai/gpt-4o")]
    rows += [as_trainee("formation-fast") for _ in range(3)]            # 1 + 3 requests in a minute: the limit is 3
    display(pd.DataFrame(rows))
    httpx.post(f"{BASE}/key/delete", headers=admin, json={"keys": [vkey]}, timeout=30)
    print("key deleted")

## 4. Tracing: every call is a Langfuse trace
Open http://localhost:13000 - login `syone@syone.com` / `syone2026`, project *Enterprise GenAI Formation*.

In [ ]:
auth = base64.b64encode(f"{os.getenv('LANGFUSE_PUBLIC_KEY','pk-lf-syone-public-key')}:{os.getenv('LANGFUSE_SECRET_KEY','sk-lf-syone-secret-key')}".encode()).decode()
resp = httpx.get("http://localhost:13000/api/public/traces?limit=5", headers={"Authorization": f"Basic {auth}"}, timeout=20)
print("Langfuse API status:", resp.status_code)
for t in resp.json().get("data", [])[:5]:
    print(f"  {t['timestamp'][:19]}  {t.get('name')}  latency={t.get('latency')}s  cost={t.get('totalCost')}")

## 5. Metrics: Prometheus scrapes the gateway and the API
Grafana (http://localhost:13001, syone / syone2026) draws them.

In [ ]:
for metric in ("litellm_requests_metric_total", "syone_requests_total"):
    q = httpx.get("http://localhost:19090/api/v1/query", params={"query": f"sum({metric})"}, timeout=10).json()
    print(metric, "=", q["data"]["result"][0]["value"][1] if q["data"]["result"] else "no data yet (send some requests first)")

### Exercises
1. Add a **second provider** to `formation-fast` (uncomment the Groq or Gemini block, put its free key in `.env`, `docker compose up -d --force-recreate litellm`). Then break the first deployment on purpose (wrong key) and watch the alias keep answering.
2. Edit the drill: put `cooldown_time: 5` and call it every 2 seconds for 20 seconds. Explain the pattern of `ms` and `fallbacks`.
3. Find your last chat in Langfuse. How many tokens did the OKF agent use compared to GraphRAG?